In [1]:
import polars as pl
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Lasso

In [2]:
def safe_json(x):
    try:
        return json.loads(x)
    except:
        return {}

def column_split(df):

    device = pd.json_normalize(df["device"])
    device = device[["isMobile", "deviceCategory"]]

    geo = pd.json_normalize(df["geoNetwork"])
    geo = geo[["subContinent", "country"]]

    totals = pd.json_normalize(df["totals"])

    totals = totals.rename(columns={"hits": "hit"})

    keep_totals = [
        "visits",
        "hit",
        "pageviews",
        "transactions",
        "transactionRevenue"
    ]

    totals = totals[keep_totals].apply(pd.to_numeric, errors="coerce")

    df_new = pd.concat([
        df.drop(columns=["device", "geoNetwork", "totals", "trafficSource"]),
        device, geo, totals
    ], axis=1)

    return df_new

def is_revenue(dataset):
    df = dataset.copy()
    df['transactionRevenue'] = pd.to_numeric(df['transactionRevenue'], errors='coerce').fillna(0)
    df['transactions'] = pd.to_numeric(df['transactions'], errors='coerce').fillna(0)
    df['has_revenue'] = (df['transactionRevenue'] > 0).astype(int)

    return df

def type_change(df):
    df['visitNumber'] = pd.to_numeric(df['visitNumber'], errors='coerce')
    df['visitStartTime'] = pd.to_datetime(df['visitStartTime'], unit='s')

    df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day

    return df

In [38]:
class TwoStageRevenueModel:
    def __init__(self, 
                 clf_params=None, 
                 reg_params=None,
                 threshold=0.5):
        
        self.clf_params = clf_params or {
            "objective": "binary",
            "boosting_type": "gbdt",

            "n_estimators": 300,
            "learning_rate": 0.05,

            "num_leaves": 31,
            "max_depth": -1,

            "subsample": 0.8,
            "colsample_bytree": 0.8,

            "class_weight": "balanced",

            "random_state": 42,
            "n_jobs": -1
        }
        
        self.reg_params = reg_params or {
            "objective": "regression",

            "n_estimators": 500,
            "learning_rate": 0.03,

            "num_leaves": 64,
            "max_depth": 8,

            "min_child_samples": 100,

            "subsample": 0.8,
            "colsample_bytree": 0.8,

            "reg_alpha": 0.1,
            "reg_lambda": 0.1,

            "random_state": 42,
            "n_jobs": -1
        }
        
        self.threshold = threshold
        
        self.clf = lgb.LGBMClassifier(**self.clf_params)
        self.reg = lgb.LGBMRegressor(**self.reg_params)
        
    # -------------------------
    # Stage 1: classification
    # -------------------------
    def fit_classifier(self, X, y):
        self.clf.fit(X, y)
        
    def predict_proba(self, X):
        return self.clf.predict_proba(X)[:, 1]
    
    # -------------------------
    # Stage 2: regression
    # -------------------------
    def fit_regressor(self, X, y):
        self.reg.fit(X, y)
        
    def predict_revenue(self, X):
        return self.clf.predict(X)
    
    # -------------------------
    # full training
    # -------------------------
    def fit(self, X, y_cls, y_reg):
        self.fit_classifier(X, y_cls)
        self.fit_regressor(X, y_reg)
        
    # -------------------------
    # final prediction
    # -------------------------
    def predict_expected_revenue(self, X):
        p_buy = self.predict_proba(X)
        rev = self.reg.predict(X)
        
        return p_buy * np.expm1(rev)
    
    # -------------------------
    # optional hard threshold version
    # -------------------------
    def predict_threshold_revenue(self, X):
        p_buy = self.predict_proba(X)
        rev = self.predict_revenue(X)
        
        return (p_buy > self.threshold) * np.expm1(rev)
    
    # -------------------------
    # evaluation
    # -------------------------
    def evaluate_classification(self, X, y):
        p = self.predict_proba(X)
        return roc_auc_score(y, p)
    
    def evaluate_regression(self, X, y):
        pred = self.predict_expected_revenue(X)
        return np.sqrt(mean_squared_error(y, pred))



In [39]:
class DataProcsser:
    def __init__(self, path_file=None):
        if path_file is None:
            self.data = None

        else:
            self.data = self.get_data(path_file)
            
        self.X_train = None
        self.X_val = None
        self.X_test = None
        self.y_train = None
        self.y_val = None
        self.y_test = None
        self.y_train_reg = None
        self.y_val_reg = None
        self.y_test_reg = None  

        self.split_data()

    # -------------------------
    # processing data
    # -------------------------
    def get_data(self,
                 file_path,
                 keep_cols = [
                     "fullVisitorId",
                     "channelGrouping",
                     "device",
                     "geoNetwork",
                     "trafficSource",
                     "visitNumber",
                     "visitStartTime",
                     "date",
                     "totals"
                 ],
                 slice_iters = 1000000
                ):
        df = pl.scan_csv(file_path,
                         schema_overrides={
                             "fullVisitorId": pl.Utf8 
                             })

        features = list(set(keep_cols) & set(df.collect_schema().names()))
        if len(features) == 0:
            print("No Features can used!!!!")
            return
        
        df = df.select(features)

        all_chunks = []
        count = 0
        for chunk in df.collect(engine="streaming").iter_slices(slice_iters):

            df_chunk = chunk.to_pandas()
            df_chunk["device"] = df_chunk["device"].apply(safe_json)
            df_chunk["geoNetwork"] = df_chunk["geoNetwork"].apply(safe_json)
            df_chunk["totals"] = df_chunk["totals"].apply(safe_json)

            df_chunk = column_split(df_chunk)
            count += len(df_chunk)

        all_chunks.append(df_chunk)

        data = pd.concat(all_chunks, axis=0)
        data = is_revenue(data)
        data = type_change(data)
        
        return data

    # -------------------------
    # split data
    # -------------------------
    def split_data(self, features=[
        "channelGrouping",
        "visitNumber",
        "isMobile",
        "deviceCategory",
        "subContinent",
        "country",
        "year",
        "month",
        "day"
    ]):
        users = self.data["fullVisitorId"].unique()
        np.random.shuffle(users)

        train_size = int(0.8 * len(users))
        val_size = int(0.1 * len(users))

        train_users = users[:train_size]
        val_users = users[train_size:train_size+val_size]
        test_users = users[train_size+val_size:]

        train_df = self.data[self.data.fullVisitorId.isin(train_users)]
        val_df   = self.data[self.data.fullVisitorId.isin(val_users)]
        test_df  = self.data[self.data.fullVisitorId.isin(test_users)]

        features = list(set(features) & set(self.data.columns))
        self.X_train = train_df[features].copy()
        self.X_val = val_df[features].copy()
        self.X_test = test_df[features].copy()

        self.y_train = train_df['has_revenue']
        self.y_val = val_df['has_revenue']
        self.y_test = test_df['has_revenue']


        cat_cols = [
            "channelGrouping",
            "isMobile",
            "deviceCategory",
            "subContinent",
            "country"
        ]

        for col in cat_cols:
            self.X_train[col] = self.X_train[col].astype("category")
            self.X_val[col] = self.X_val[col].astype("category")
            self.X_test[col] = self.X_test[col].astype("category")

        self.y_train_reg = np.log1p(train_df["transactionRevenue"])
        self.y_val_reg = np.log1p(val_df["transactionRevenue"])
        self.y_test_reg = np.log1p(test_df["transactionRevenue"])

In [9]:
df = DataProcsser('./data/train_v2.csv')

In [10]:
df.data.head()

,fullVisitorId,date,visitStartTime,visitNumber,channelGrouping,isMobile,deviceCategory,subContinent,country,visits,hit,pageviews,transactions,transactionRevenue,has_revenue,year,month,day
0,6821979272477799804,2018-02-23,2018-02-24 02:37:24,1,Organic Search,False,desktop,Northern America,United States,1,39,30.0,0.0,0.0,0,2018,2,23
1,3294419200785362461,2018-02-23,2018-02-23 20:34:46,3,Referral,False,desktop,Northern America,United States,1,39,30.0,0.0,0.0,0,2018,2,23
2,4021719711274238069,2018-02-23,2018-02-24 06:48:40,2,Organic Search,True,tablet,Australasia,Australia,1,40,29.0,0.0,0.0,0,2018,2,23
3,9641057345301469522,2018-02-23,2018-02-23 18:00:00,1,Organic Search,False,desktop,Northern America,United States,1,41,30.0,0.0,0.0,0,2018,2,23
4,7058072212478304786,2018-02-23,2018-02-23 22:54:06,2,Referral,False,desktop,Northern America,United States,1,41,28.0,0.0,0.0,0,2018,2,23


In [11]:
df.X_test.head()

,day,deviceCategory,country,month,isMobile,visitNumber,year,channelGrouping,subContinent
7,23,desktop,Peru,2,False,1,2018,Organic Search,South America
12,23,tablet,Canada,2,True,1,2018,Organic Search,Northern America
29,23,mobile,Libya,2,True,1,2018,Social,Northern Africa
35,23,mobile,Pakistan,2,True,1,2018,Social,Southern Asia
38,23,mobile,India,2,True,1,2018,Social,Southern Asia


In [40]:
model = TwoStageRevenueModel()

In [41]:
model.fit_classifier(df.X_train, df.y_train)

[LightGBM] [Info] Number of positive: 6167, number of negative: 559673
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027036 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 443
[LightGBM] [Info] Number of data points in the train set: 565840, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


In [42]:
print("Validation AUC:", model.evaluate_classification(df.X_val, df.y_val))
print("Test AUC:", model.evaluate_classification(df.X_test, df.y_test))

Validation AUC: 0.8876459473052799
Test AUC: 0.8851033107089812


In [43]:
model.fit_regressor(df.X_train, df.y_train_reg)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027668 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 443
[LightGBM] [Info] Number of data points in the train set: 565840, number of used features: 9
[LightGBM] [Info] Start training from score 0.193479
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [44]:
print("Val RMSE: ", model.evaluate_regression(df.X_val, df.y_val_reg) )
print("Test RMSE: ", model.evaluate_regression(df.X_test, df.y_test_reg) )

Val RMSE:  1.894905606622181
Test RMSE:  1.9231433587060072


In [45]:
model.predict_revenue(df.X_val)

array([1, 0, 0, ..., 0, 0, 0])